In [1]:
import re
import nltk
import fasttext
from nltk.corpus import stopwords

import numpy as np
import pandas as pd

from keras.models import Model
from keras.utils import pad_sequences
from keras.preprocessing.text import Tokenizer
from keras.layers import Dense, Dropout, Embedding, Input, LSTM

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score

In [2]:
Max_sequence_length = 50
Embedding_dim = 100

In [3]:
df = pd.read_csv('CrowdFlower_Dataset.csv')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   tweet_id   40000 non-null  int64 
 1   sentiment  40000 non-null  object
 2   author     40000 non-null  object
 3   content    40000 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.2+ MB


In [5]:
data = df.sample(n = 20000)

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 20000 entries, 4422 to 35507
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   tweet_id   20000 non-null  int64 
 1   sentiment  20000 non-null  object
 2   author     20000 non-null  object
 3   content    20000 non-null  object
dtypes: int64(1), object(3)
memory usage: 781.2+ KB


In [7]:
data.head()

,tweet_id,sentiment,author,content
4422,1960354251,worry,liilaantunes,"waking now, so lazy and very worry ..."
8558,1962218483,neutral,RialtoCafe,"@artbynemo My name is Chad, but I won't be her..."
28380,1696178078,happiness,meganob,It is going to be a beautiful day!
11126,1963125836,sadness,Ciara101,I really wanted that job
26837,1695546908,love,MMR04,x]loveyoutoo!


In [8]:
wl = nltk.WordNetLemmatizer()
ps = nltk.PorterStemmer()

In [9]:
def preprocess(text):
    text = text.lower()
    text = re.sub('[^a-zA-Z\s]','',text)
    stop = stopwords.words('english')
    text = [ps.stem(wl.lemmatize(word)) for word in text.split() if word not in stop]
    return text

In [10]:
processed_data = data['content'].map(preprocess)

In [11]:
processed_data.head()

4422                         [wake, lazi, worri]
8558     [artbynemo, name, chad, wont, tomorrow]
28380                          [go, beauti, day]
11126                        [realli, want, job]
26837                              [xloveyoutoo]
Name: content, dtype: object

In [16]:
data['sentiment'].value_counts()

neutral       4328
worry         4213
happiness     2599
sadness       2536
love          1932
surprise      1071
fun            871
relief         777
hate           689
empty          447
enthusiasm     379
boredom         98
anger           60
Name: sentiment, dtype: int64

In [22]:
# data.loc[data['sentiment'] == 'enthusiasm','sentiment'] = 'happiness'
# data.loc[data['sentiment'] == 'empty','sentiment'] = 'sadness'
# data.loc[data['sentiment'] == 'boredom','sentiment'] = 'sadness'
# data.loc[data['sentiment'] == 'fun','sentiment'] = 'happiness'
# data.loc[data['sentiment'] == 'hate','sentiment'] = 'anger'
# data.loc[data['sentiment'] == 'anger','sentiment'] = 'sadness'
# data.loc[data['sentiment'] == 'surprise','sentiment'] = 'love'
# data.loc[data['sentiment'] == 'relief','sentiment'] = 'love'
# data['sentiment'].value_counts()

neutral      4328
worry        4213
happiness    3849
sadness      3830
love         3780
Name: sentiment, dtype: int64

In [23]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(processed_data)
sequences = tokenizer.texts_to_sequences(processed_data)

word_index = tokenizer.word_index
len(word_index)

27201

In [24]:
features = pad_sequences(sequences, Max_sequence_length)
labels = pd.get_dummies(data['sentiment'], dtype='int')

features.shape, labels.shape

((20000, 50), (20000, 5))

In [25]:
labels.head()

,happiness,love,neutral,sadness,worry
4422,0,0,0,0,1
8558,0,0,1,0,0
28380,1,0,0,0,0
11126,0,0,0,1,0
26837,0,1,0,0,0


In [26]:
x_train, x_test , y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=5)
x_val, x_test, y_val, y_test = train_test_split(features, labels, test_size=0.5, random_state=7)

In [27]:
embedding_model = fasttext.load_model(r"D:\Notebook\Projects\Embedding_Models\fasttext_embedding_model.bin")
        
embedding_matrix = np.zeros((len(word_index)+1, Embedding_dim))
for word,i in word_index.items():
    vector = embedding_model.get_word_vector(word)
    if vector is not None:
        embedding_matrix[i] = vector
        
        
embedding_layer = Embedding(len(word_index)+1,Embedding_dim, weights = [embedding_matrix],
                            input_length=Max_sequence_length)(input_sequences)
        

In [31]:
convs = []
filter_sizes = [3,4,5]

sequence_input = Input(shape=(MAX_SEQUENCE_LENGTH,))
embedded_sequences = embedding_layer(sequence_input)

for fsz in filter_sizes:
    l_conv = Conv1D(128,fsz,activation='relu')(embedded_sequences)
    l_pool = MaxPooling1D(5)(l_conv)
    convs.append(l_pool)   
l_merge = Concatenate()(convs)
l_cov1= Conv1D(filters=128, kernel_size=5, activation='relu')(l_merge)
l_pool1 = MaxPooling1D(5)(l_cov1)
# l_cov2 = Conv1D(filters=128, kernel_size=5, activation='relu')(l_pool1)
# l_pool2 = MaxPooling1D(30)(l_cov2)
l_flat = Flatten()(l_pool1)
l_dense = Dense(128, activation='relu')(l_flat)
preds = Dense(13, activation='softmax')(l_dense)

model = Model(sequence_input, preds)
model.compile(loss='binary_crossentropy',
              optimizer='Nadam',
              metrics=['acc'])

model.summary()


In [32]:
model.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 50)]              0         
                                                                 
 embedding_1 (Embedding)     (None, 50, 100)           2720200   
                                                                 
 lstm_3 (LSTM)               (None, 50, 256)           365568    
                                                                 
 dropout_1 (Dropout)         (None, 50, 256)           0         
                                                                 
 lstm_4 (LSTM)               (None, 50, 128)           197120    
                                                                 
 lstm_5 (LSTM)               (None, 64)                49408     
                                                                 
 dense_1 (Dense)             (None, 5)                 325 

In [ ]:
history = model.fit(x_train, y_train, validation_data=(x_val,y_val), epochs=25, batch_size = 150, verbose=1)

Epoch 1/25
107/107 [==============================] - 106s 804ms/step - loss: 1.5298 - acc: 0.3134 - val_loss: 1.4545 - val_acc: 0.3523
Epoch 2/25
107/107 [==============================] - 68s 636ms/step - loss: 1.4142 - acc: 0.3921 - val_loss: 1.3443 - val_acc: 0.4332
Epoch 3/25
 19/107 [====>.........................] - ETA: 34s - loss: 1.2994 - acc: 0.4554

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline 
# list all data in history
print(history.history.keys())
# summarize history for accuracy
plt.plot(history.history['acc'])
plt.plot(history.history['val_acc'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()
# summarize history for loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [34]:
y_pred = model.predict(x_test, verbose=1)

79/79 [==============================] - 5s 52ms/step


In [37]:
def fx(array):
    return np.argmax(array)

temp = np.array(y_test)
st_pred = y_pred.tolist()
st_temp = temp.tolist()

arr = np.array(list(map(fx,st_temp)))
ans = np.array(list(map(fx,st_pred)))


In [ ]:
acc = accuracy_score(arr,ans)
acc

In [39]:
f1 = f1_score(arr,ans, zero_division=1.0, average=None)
r1 = recall_score(arr,ans, zero_division=1.0,average=None)
p1 = precision_score(arr,ans, zero_division=1.0, average=None)
print(f1)
print(r1)
print(p1)

[0.         1.         0.60344828 0.60606061 0.7257384  0.80728376
 0.59649123 0.82978723 0.84955752 0.67619048 0.802589   0.65605096
 0.81578947]
[1.         0.         0.48611111 0.625      0.66153846 0.80120482
 0.57303371 0.85903084 0.89256198 0.65740741 0.84641638 0.59537572
 0.80073801]
[0.         0.         0.79545455 0.58823529 0.80373832 0.81345566
 0.62195122 0.80246914 0.81050657 0.69607843 0.76307692 0.73049645
 0.83141762]


In [40]:
f1 = f1_score(arr,ans, zero_division=1.0, average='micro')
r1 = recall_score(arr,ans, zero_division=1.0,average='micro')
p1 = precision_score(arr,ans, zero_division=1.0, average='micro')
print(f1)
print(r1)
print(p1)

0.7803999999999999
0.7804
0.7804


In [41]:
f1 = f1_score(arr,ans, zero_division=1.0, average='macro')
r1 = recall_score(arr,ans, zero_division=1.0,average='macro')
p1 = precision_score(arr,ans, zero_division=1.0, average='macro')
print(f1)
print(r1)
print(p1)

0.6899220713933782
0.6768014184500556
0.6351446284316619


In [42]:
f1 = f1_score(arr,ans, zero_division=1.0,average='weighted')
r1 = recall_score(arr,ans, zero_division=1.0,average='weighted')
p1 = precision_score(arr,ans, zero_division=1.0,average='weighted')
print(f1)
print(r1)
print(p1)

0.7813522022240165
0.7804
0.7862491438605635
